# DeepSeek-OCR on Google Colab

1. Select **Runtime → Change runtime type → GPU**.
2. Run the cells from top to bottom through the smoke test. The first run downloads the model weights (about 6.7 GB).
3. The notebook clones and checks [this public repository](https://github.com/ubaid-148/deeksheekocr), then runs OCR on a simple test image. The last cell lets you upload your own image.

Inference follows the [upstream vLLM DeepSeek-OCR recipe](https://docs.vllm.ai/projects/recipes/en/latest/DeepSeek/DeepSeek-OCR.html). Colab GPU availability varies; if no GPU is assigned, reconnect with a GPU runtime.

In [ ]:
import shutil
import subprocess

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Select a GPU runtime: Runtime → Change runtime type → GPU')
subprocess.run(['nvidia-smi'], check=True)

## Clone and check the public repository

The repository has Python scripts rather than a compiled build target. This cell checks every Python source file without creating build files.

In [ ]:
import ast
from pathlib import Path

REPO_URL = 'https://github.com/ubaid-148/deeksheekocr.git'
REPO_DIR = Path('/content/deeksheekocr')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

sources = sorted(REPO_DIR.rglob('*.py'))
if not sources:
    raise RuntimeError('No Python files found in the public repository')
for source in sources:
    ast.parse(source.read_text(encoding='utf-8'), filename=str(source))
print(f'Checked {len(sources)} Python files in {REPO_DIR}')
subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--short', '--branch'], check=True)

## Install the GPU runtime

vLLM supplies the model runtime and compatible PyTorch packages. This notebook uses vLLM's CUDA backend selection, so the pinned `requirements.txt` for the repository's older Transformers example is not installed here. Imports are checked in a fresh Python process; this avoids stale Pillow modules already loaded by Colab.

In [ ]:
import importlib.metadata
import sys

try:
    vllm_version = importlib.metadata.version('vllm')
except importlib.metadata.PackageNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--system', '-U', 'vllm', '--torch-backend=auto'], check=True)
    vllm_version = importlib.metadata.version('vllm')

import_check = [sys.executable, '-c', 'from PIL import Image, ImageText; from vllm import LLM, SamplingParams']
checked = subprocess.run(import_check, capture_output=True, text=True)
if checked.returncode != 0 and "cannot import name '_Ink'" in checked.stderr:
    print('Repairing mixed Pillow files on disk')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'pillow'], check=True)
    checked = subprocess.run(import_check, capture_output=True, text=True)
if checked.returncode != 0:
    raise RuntimeError('Fresh-process import failed:\n' + checked.stderr)
print('vLLM', vllm_version, 'and Pillow import correctly in a fresh process')

## Choose the GPU dtype

T4 GPUs need float16. A100 and newer GPUs use bfloat16. OCR runs through this repository's `colab_infer.py` in a fresh process, so Colab's cached Pillow modules cannot affect it.

In [ ]:
probe = subprocess.check_output([
    sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(); print(*torch.cuda.get_device_capability(0), sep=".")'
], text=True).strip()
gpu_major, gpu_minor = map(int, probe.splitlines()[-1].split('.'))
if (gpu_major, gpu_minor) < (7, 5):
    raise RuntimeError('vLLM needs an NVIDIA GPU with compute capability 7.5 or newer')
model_dtype = 'float16' if gpu_major < 8 else 'bfloat16'
print('GPU compute capability:', f'{gpu_major}.{gpu_minor}', 'model dtype:', model_dtype)
runner = REPO_DIR / 'colab_infer.py'
if not runner.is_file():
    raise FileNotFoundError(f'Colab runner missing from repository: {runner}')

## Smoke test

In [ ]:
smoke_output = Path('/content/deepseek_ocr_smoke.md')
subprocess.run([
    sys.executable, str(runner), '--smoke', '--dtype', model_dtype,
    '--output', str(smoke_output),
], check=True)
print('Smoke test saved to', smoke_output)

## Run OCR on your own image (optional)

Run this cell separately after the smoke test. Upload a PNG, JPG, or JPEG file. The Markdown result is saved and downloaded.

In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one PNG, JPG, or JPEG image')
image_name = next(iter(uploaded))
if Path(image_name).suffix.lower() not in {'.png', '.jpg', '.jpeg'}:
    raise ValueError('Only PNG, JPG, and JPEG are supported in this cell')
output_file = Path('/content') / f'{Path(image_name).stem}_ocr.md'
subprocess.run([
    sys.executable, str(runner), '--image', image_name, '--dtype', model_dtype,
    '--output', str(output_file),
], check=True)
files.download(str(output_file))